In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import numpy as np
import scipy.io as sio
import torch
from torch.utils.data import Dataset, DataLoader


Mounted at /content/drive




# Model Architecture



In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class HybridTransformerLayer(nn.Module):
    """
    One Transformer layer with:
    - Multi-head self-attention (global interactions)
    - Feed-forward network (per-token non-linearity)
    - Conv1d branch over the token sequence (local smoothing over neighbouring tokens)
    """
    def __init__(self, d_model=64, n_heads=4, ff_dim=128,
                 conv_kernel_size=3, dropout=0.1):
        super().__init__()

        # Multi-head self-attention (batch_first=True -> input (B, S, D))
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True
        )

        # LayerNorm before attention (pre-LN style is more stable)
        self.attn_norm = nn.LayerNorm(d_model)

        # Feed-forward network (position-wise)
        self.ffn = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model),
            nn.Dropout(dropout),
        )

        # Conv branch over the token sequence.
        # We view the sequence length as "time" and embedding dim as channels.
        # Input to Conv1d: (B, D, S)  -> output: (B, D, S)
        padding = conv_kernel_size // 2
        self.conv_branch = nn.Sequential(
            nn.Conv1d(
                in_channels=d_model,
                out_channels=d_model,
                kernel_size=conv_kernel_size,
                padding=padding
            ),
            nn.BatchNorm1d(d_model),
            nn.ELU()
        )

    def forward(self, x):
        """
        x: (B, S, D)  B=batch, S=sequence length (22 tokens), D=embedding dim
        returns: (B, S, D)
        """

        # ---- 1. Self-attention (global) ----
        # Pre-norm
        x_norm = self.attn_norm(x)
        attn_out, _ = self.self_attn(x_norm, x_norm, x_norm)  # (B, S, D)
        x = x + attn_out  # residual connection

        # ---- 2. Feed-forward (per token) ----
        ffn_out = self.ffn(x)   # LayerNorm is inside ffn
        x = x + ffn_out         # residual

        # ---- 3. Conv branch (local over tokens) ----
        # Treat seq dimension as 1D "time": (B, S, D) -> (B, D, S)
        x_conv = x.transpose(1, 2)          # (B, D, S)
        x_conv = self.conv_branch(x_conv)   # (B, D, S)
        x_conv = x_conv.transpose(1, 2)     # (B, S, D)

        # Add conv branch as another residual
        x = x + x_conv

        return x


class SpatioSpectroEncoder(nn.Module):
    def __init__(self, d_model=64, n_heads=4, n_layers=2,
                 ff_dim=128, cnn_channels=16,
                 conv_kernel_size=3, dropout=0.1,
                 channel_drop_prob=0.0):
        super().__init__()
        self.d_model = d_model
        self.channel_drop_prob = channel_drop_prob

        # --- CNN front-end ---
        self.cnn_front = nn.Sequential(
            nn.Conv2d(1, cnn_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(cnn_channels),
            nn.ReLU(),
            nn.Conv2d(cnn_channels, cnn_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(cnn_channels),
            nn.ReLU(),
        )

        # --- Channel / band MLPs ---
        self.channel_mlp = nn.Sequential(
            nn.Linear(5, d_model),
            nn.ReLU(),
        )
        self.band_mlp = nn.Sequential(
            nn.Linear(17, d_model),
            nn.ReLU(),
        )

        # --- Type + pos embeddings ---
        self.type_embedding = nn.Embedding(2, d_model)
        self.max_tokens = 22
        self.pos_embedding = nn.Embedding(self.max_tokens, d_model)

        # --- Hybrid Transformer layers ---
        self.layers = nn.ModuleList([
            HybridTransformerLayer(
                d_model=d_model,
                n_heads=n_heads,
                ff_dim=ff_dim,
                conv_kernel_size=conv_kernel_size,
                dropout=dropout,
            )
            for _ in range(n_layers)
        ])

    def forward(self, x):
        """
        x: (B, 17, 5)  single time frame
        returns:
            frame_emb: (B, d_model)
            enc_out:   (B, 22, d_model)  # for GradCAM
        """
        B = x.size(0)

        # optional channel-drop
        if self.training and self.channel_drop_prob > 0.0:
            drop_mask = torch.rand(B, x.size(1), 1, device=x.device)
            drop_mask = (drop_mask > self.channel_drop_prob).float()
            x = x * drop_mask

        # CNN front-end
        x_img = x.unsqueeze(1)                # (B, 1, 17, 5)
        x_cnn = self.cnn_front(x_img)        # (B, Cc, 17, 5)
        x_cnn = x_cnn.mean(dim=1)            # (B, 17, 5)

        # Channel tokens
        ch_tokens = self.channel_mlp(x_cnn)  # (B, 17, D)

        # Band tokens
        x_band = x_cnn.permute(0, 2, 1)      # (B, 5, 17)
        band_tokens = self.band_mlp(x_band)  # (B, 5, D)

        tokens = torch.cat([ch_tokens, band_tokens], dim=1)  # (B, 22, D)

        # Type embeddings
        type_ids = torch.cat([
            torch.zeros(17, dtype=torch.long),
            torch.ones(5, dtype=torch.long)
        ], dim=0).to(x.device)              # (22,)
        type_ids = type_ids.unsqueeze(0).expand(B, -1)       # (B, 22)
        type_emb = self.type_embedding(type_ids)

        # Pos embeddings
        seq_len = tokens.size(1)
        pos_ids = torch.arange(seq_len, device=x.device).long()  # (22,)
        pos_ids = pos_ids.unsqueeze(0).expand(B, -1)             # (B, 22)
        pos_emb = self.pos_embedding(pos_ids)

        x_tok = tokens + type_emb + pos_emb  # (B, 22, D)

        for layer in self.layers:
            x_tok = layer(x_tok)             # (B, 22, D)

        enc_out = x_tok
        frame_emb = enc_out.mean(dim=1)      # (B, D)

        return frame_emb, enc_out


class SpatioSpectroTemporalNet(nn.Module):
    """
    Spatio-spectro-temporal model.

    Input:
        x: (B, T, 17, 5)  -> T frames of DE features per sample
    Output:
        logits:    (B, num_classes)   # or regression dims
        token_seq: (B, T, 22, D)      # token embeddings for GradCAM
    """
    def __init__(self,
                 d_model=64,
                 temporal_hidden=64,
                 num_layers_rnn=1,
                 num_classes=3,
                 use_bi=True,
                 dropout=0.1,
                 channel_drop_prob=0.0):
        super().__init__()

        # Your existing per-frame encoder (no head)
        self.frame_encoder = SpatioSpectroEncoder(
            d_model=d_model,
            n_heads=4,
            n_layers=2,
            ff_dim=128,
            cnn_channels=16,
            conv_kernel_size=3,
            dropout=dropout,
            channel_drop_prob=channel_drop_prob,
        )

        self.use_bi = use_bi
        self.temporal_hidden = temporal_hidden

        # Temporal backbone: GRU over frame embeddings
        self.rnn = nn.GRU(
            input_size=d_model,
            hidden_size=temporal_hidden,
            num_layers=num_layers_rnn,
            batch_first=True,
            bidirectional=use_bi,
        )

        rnn_out_dim = temporal_hidden * (2 if use_bi else 1)

        # Final head
        self.head = nn.Sequential(
            nn.Linear(rnn_out_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)   # or 1/2 for regression
        )

    def forward(self, x):
        """
        x: (B, T, 17, 5)
        returns:
            logits:    (B, num_classes)
            token_seq: (B, T, 22, D)
        """
        B, T, C, F = x.shape

        # ---- 1) Encode each frame independently ----
        x_flat = x.view(B * T, C, F)           # (B*T, 17, 5)

        frame_emb_flat, token_flat = self.frame_encoder(x_flat)
        # frame_emb_flat: (B*T, D)
        # token_flat:     (B*T, 22, D)

        D = frame_emb_flat.size(-1)

        # ---- 2) Reshape back to sequences ----
        frame_emb_seq = frame_emb_flat.view(B, T, D)       # (B, T, D)
        token_seq = token_flat.view(B, T, 22, D)           # (B, T, 22, D)

        # ---- 3) Temporal GRU over frame embeddings ----
        rnn_out, _ = self.rnn(frame_emb_seq)               # (B, T, rnn_out_dim)

        # Temporal pooling (mean over time, or you can use last step / attention)
        pooled = rnn_out.mean(dim=1)                       # (B, rnn_out_dim)

        logits = self.head(pooled)                         # (B, num_classes)

        return logits, token_seq

# Dataset

In [26]:
import os, glob
import numpy as np
from scipy.io import loadmat
import torch
from torch.utils.data import Dataset, DataLoader

# ======================
# 1. Load & stack data
# ======================

def load_seedvig_5band(base_dir, feature_key='de_LDS', label_candidates=None):
    if label_candidates is None:
        label_candidates = ['PERCLOS', 'perclos', 'label', 'labels', 'y']

    eeg_dir   = os.path.join(base_dir, 'EEG_Feature_5Bands')
    label_dir = os.path.join(base_dir, 'perclos_labels')

    eeg_files   = sorted(glob.glob(os.path.join(eeg_dir, '*.mat')))
    label_files = sorted(glob.glob(os.path.join(label_dir, '*.mat')))

    all_X, all_y, all_subj = [], [], []

    for subj_id, (eeg_path, label_path) in enumerate(zip(eeg_files, label_files)):
        eeg_mat = loadmat(eeg_path)
        if feature_key not in eeg_mat:
            continue

        X = eeg_mat[feature_key]        # (17, n_samples, 5)
        if X.ndim != 3 or X.shape[0] != 17 or X.shape[2] != 5:
            continue

        # (17, n, 5) -> (n, 17, 5)
        X = np.transpose(X, (1, 0, 2))

        label_mat = loadmat(label_path)
        label_key = next((k for k in label_candidates if k in label_mat), None)
        if label_key is None:
            continue

        y = label_mat[label_key].squeeze()   # (n_samples,)
        if len(y) != X.shape[0]:
            continue

        all_X.append(X)
        all_y.append(y)
        all_subj.append(np.full(len(y), subj_id, dtype=int))

    if not all_X:
        raise RuntimeError("No valid subjects found. Check paths/keys.")

    X_all    = np.concatenate(all_X, axis=0)       # (N, 17, 5)
    y_all    = np.concatenate(all_y, axis=0)       # (N,)
    subj_all = np.concatenate(all_subj, axis=0)    # (N,)

    return X_all, y_all, subj_all


BASE_DIR = '/content/drive/MyDrive/CSE465:EEG/SEED-VIG'
X_all, y_all, subj_all = load_seedvig_5band(BASE_DIR, feature_key='de_LDS')

print("Raw shapes:", X_all.shape, y_all.shape, subj_all.shape)

# ======================
# 2. Normalize features
# ======================

X_norm = np.zeros_like(X_all)
for s in np.unique(subj_all):
    idx = np.where(subj_all == s)[0]
    X_s = X_all[idx]                        # (Ns, 17, 5)
    X_flat = X_s.reshape(X_s.shape[0], -1)
    mean   = X_flat.mean(axis=0, keepdims=True)
    std    = X_flat.std(axis=0, keepdims=True) + 1e-8
    X_norm[idx] = ((X_flat - mean) / std).reshape(X_s.shape[0], 17, 5)

X_all = X_norm


# ======================
# 3. Binary labels
# ======================

PERCLOS_THRESH = 0.4
y_bin_all = (y_all >= PERCLOS_THRESH).astype(int)
print("Binary label distribution (all):", np.bincount(y_bin_all))

# ======================
# 4. Subject-wise split
# ======================

rng = np.random.default_rng(42)
unique_subj = np.unique(subj_all)
rng.shuffle(unique_subj)

n_subj   = len(unique_subj)
n_train  = int(0.7 * n_subj)
n_val    = int(0.15 * n_subj)
n_test   = n_subj - n_train - n_val   # should match ~15%

train_subj = unique_subj[:n_train]
val_subj   = unique_subj[n_train:n_train+n_val]
test_subj  = unique_subj[n_train+n_val:]

print("\n=== Subject split (IDs) ===")
print(f"Train subjects ({len(train_subj)}): {train_subj}")
print(f"Val subjects   ({len(val_subj)}): {val_subj}")
print(f"Test subjects  ({len(test_subj)}): {test_subj}")
print(f"Val/Test subject count equal? {len(val_subj) == len(test_subj)}")

train_mask = np.isin(subj_all, train_subj)
val_mask   = np.isin(subj_all, val_subj)
test_mask  = np.isin(subj_all, test_subj)

X_train, y_train = X_all[train_mask], y_bin_all[train_mask]
X_val,   y_val   = X_all[val_mask],   y_bin_all[val_mask]
X_test,  y_test  = X_all[test_mask],  y_bin_all[test_mask]

print("\n#Train:", X_train.shape[0], "#Val:", X_val.shape[0], "#Test:", X_test.shape[0])

print("\n=== Binary label distribution per split (frame-level) ===")
print("Train:", np.bincount(y_train))
print("Val:  ", np.bincount(y_val))
print("Test: ", np.bincount(y_test))

subj_train = subj_all[train_mask]
subj_val   = subj_all[val_mask]
subj_test  = subj_all[test_mask]

# ======================
# 5. Dataset + loaders (temporal)
# ======================

from torch.utils.data import Dataset

class SEEDVIGBinarySeqDataset(Dataset):
    def __init__(self, X, y_bin, subj_ids, T_seq=10, step=1):
        """
        X:        (N, 17, 5)
        y_bin:    (N,) 0/1
        subj_ids: (N,) subject id per frame
        T_seq:    temporal sequence length (e.g., 10 frames)
        step:     stride between sequence starts (e.g., 1 for max overlap)
        """
        self.T_seq = T_seq
        self.step = step

        seq_X = []
        seq_y = []

        # Ensure everything is numpy
        X = np.asarray(X)
        y_bin = np.asarray(y_bin)
        subj_ids = np.asarray(subj_ids)

        # Build sequences within each subject separately
        for s in np.unique(subj_ids):
            idx = np.where(subj_ids == s)[0]
            idx = np.sort(idx)   # just to be safe; should already be in order

            n = len(idx)
            if n < T_seq:
                continue  # not enough frames for this subject

            for start in range(0, n - T_seq + 1, step):
                window_idx = idx[start:start + T_seq]     # length T_seq

                labels_window = y_bin[window_idx]
                ones = np.sum(labels_window == 1)
                zeros = len(labels_window) - ones
                if ones == 0 or zeros == 0:
                    # pure windows
                    seq_y.append(labels_window[0])
                else:
                    # majority label
                    seq_y.append(1 if ones > zeros else 0)

                seq_X.append(X[window_idx])

                #labels_window = y_bin[window_idx]
                #if np.all(labels_window == 0) or np.all(labels_window == 1):
                #    seq_X.append(X[window_idx])
                #    seq_y.append(labels_window[0])
                #else:
                    # skip mixed windows to reduce label noise
                #    continue


        if len(seq_X) == 0:
            raise RuntimeError("No sequences could be built. "
                               "Try smaller T_seq or check subj_ids alignment.")

        seq_X = np.stack(seq_X, axis=0)          # (M, T_seq, 17, 5)
        seq_y = np.array(seq_y)                  # (M,)

        self.X = torch.from_numpy(seq_X).float()
        self.y = torch.from_numpy(seq_y).long()

        print(f"[SEEDVIGBinarySeqDataset] Built {self.X.shape[0]} sequences of "
              f"length T={T_seq} from {len(np.unique(subj_ids))} subjects.")

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        # returns x: (T_seq, 17, 5), y: scalar
        return self.X[idx], self.y[idx]


T_SEQ = 20   # or 20, tune this
STEP  = 5   # stride for sliding window

train_ds = SEEDVIGBinarySeqDataset(X_train, y_train, subj_train, T_seq=T_SEQ, step=STEP)
val_ds   = SEEDVIGBinarySeqDataset(X_val,   y_val,   subj_val,   T_seq=T_SEQ, step=STEP)
test_ds  = SEEDVIGBinarySeqDataset(X_test,  y_test,  subj_test,  T_seq=T_SEQ, step=STEP)

print("\n=== Binary label distribution per split (sequence-level) ===")
print("Train seq:", np.bincount(train_ds.y.numpy()))
print("Val seq:  ", np.bincount(val_ds.y.numpy()))
print("Test seq: ", np.bincount(test_ds.y.numpy()))

BATCH_SIZE = 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print("\nExample batch shapes:")
xb, yb = next(iter(train_loader))
print("xb:", xb.shape, "yb:", yb.shape)  # expect xb: (B, T_SEQ, 17, 5)

print("Batches:", len(train_loader), "train |", len(val_loader), "val |", len(test_loader), "test")


Raw shapes: (20355, 17, 5) (20355,) (20355,)
Binary label distribution (all): [ 9318 11037]

=== Subject split (IDs) ===
Train subjects (16): [18 10 17 22 16 15  7  6  9  3  0 19 12  5 11 14]
Val subjects   (3): [21  2  4]
Test subjects  (4): [20  1 13  8]
Val/Test subject count equal? False

#Train: 14160 #Val: 2655 #Test: 3540

=== Binary label distribution per split (frame-level) ===
Train: [5999 8161]
Val:   [1364 1291]
Test:  [1955 1585]
[SEEDVIGBinarySeqDataset] Built 2784 sequences of length T=20 from 16 subjects.
[SEEDVIGBinarySeqDataset] Built 522 sequences of length T=20 from 3 subjects.
[SEEDVIGBinarySeqDataset] Built 696 sequences of length T=20 from 4 subjects.

=== Binary label distribution per split (sequence-level) ===
Train seq: [1185 1599]
Val seq:   [265 257]
Test seq:  [381 315]

Example batch shapes:
xb: torch.Size([128, 20, 17, 5]) yb: torch.Size([128])
Batches: 22 train | 5 val | 6 test


# Train util

In [27]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha  # tensor of shape [num_classes] or None
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        # logits: (B, C), targets: (B,)
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce)           # pt = prob of the true class
        loss = (1 - pt) ** self.gamma * ce

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        else:
            return loss


In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm


def train_and_evaluate_classification(
    model,
    train_loader,
    val_loader,
    test_loader,
    y_train,
    num_epochs=30,
    lr=1e-3,
    weight_decay=1e-4,
    checkpoint_path="sstnet_cls_best.pth",
    device=None
):
    """
    Generic training loop for classification.

    Assumptions:
      - model(x) -> (logits, token_seq)
      - x from loaders has shape (B, T, 17, 5)
      - y_train and all loader labels are encoded as integers in [0, K-1]
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)
    model = model.to(device)

    # ----- Class info & weights -----
    classes = np.unique(y_train)
    n_classes = len(classes)
    if classes.min() != 0 or classes.max() != n_classes - 1:
        raise ValueError(
            f"y_train must be encoded as 0..K-1. Got min={classes.min()}, max={classes.max()}."
        )
    print(f"Detected {n_classes} classes: {classes}")

    class_counts = np.bincount(y_train)
    class_weights = class_counts.sum() / (len(class_counts) * class_counts)
    class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
    #criterion = nn.CrossEntropyLoss(weight=class_weights)
    criterion = FocalLoss(alpha=class_weights, gamma=2.0)


    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # F1 averaging mode: binary for 2-class, macro for multi-class
    f1_avg = "binary" if n_classes == 2 else "macro"

    best_val_f1 = 0.0

    for epoch in range(1, num_epochs + 1):
        # ----- TRAIN -----
        model.train()
        train_losses = []

        for xb, yb in tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs} [Train]", leave=False):
            # xb: (B, T, 17, 5)
            xb = xb.to(device)
            yb = yb.to(device).long()   # (B,)

            optimizer.zero_grad()
            logits, _ = model(xb)       # logits: (B, n_classes)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        avg_train_loss = np.mean(train_losses)

        # ----- VALIDATION -----
        model.eval()
        all_true = []
        all_pred = []
        val_losses = []

        with torch.no_grad():
            for xb, yb in tqdm(val_loader, desc=f"Epoch {epoch}/{num_epochs} [Val]", leave=False):
                xb = xb.to(device)
                yb = yb.to(device).long()

                logits, _ = model(xb)
                loss = criterion(logits, yb)
                val_losses.append(loss.item())

                preds = logits.argmax(dim=-1)

                all_true.extend(yb.cpu().numpy())
                all_pred.extend(preds.cpu().numpy())

        all_true = np.array(all_true)
        all_pred = np.array(all_pred)

        val_acc = accuracy_score(all_true, all_pred)
        val_f1  = f1_score(all_true, all_pred, average=f1_avg)
        avg_val_loss = np.mean(val_losses)

        print(
            f"Epoch {epoch:03d} | "
            f"TrainLoss={avg_train_loss:.4f} | "
            f"ValLoss={avg_val_loss:.4f} | "
            f"ValACC={val_acc*100:.2f}% | ValF1({f1_avg})={val_f1:.3f}"
        )

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), checkpoint_path)
            print(f"  👉 New best model saved (ValF1={best_val_f1:.3f})")

    # ----- TEST (load best) -----
    print("\nLoading best model from:", checkpoint_path)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)
    model.eval()

    all_true = []
    all_pred = []

    with torch.no_grad():
        for xb, yb in tqdm(test_loader, desc="Testing", leave=False):
            xb = xb.to(device)
            yb = yb.to(device).long()

            logits, _ = model(xb)
            preds = logits.argmax(dim=-1)

            all_true.extend(yb.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())

    all_true = np.array(all_true)
    all_pred = np.array(all_pred)

    test_acc = accuracy_score(all_true, all_pred)
    test_f1  = f1_score(all_true, all_pred, average=f1_avg)

    print(f"\nTestACC={test_acc*100:.2f}% | TestF1({f1_avg})={test_f1:.3f}")
    return {"ACC": test_acc, "F1": test_f1}


# Training

In [5]:
import numpy as np

# Number of classes from labels
n_classes = len(np.unique(y_train))
model = SpatioSpectroTemporalNet(
    d_model=64,
    temporal_hidden=64,
    num_layers_rnn=1,
    num_classes=n_classes,   # make sure this matches your labels
    use_bi=True,
    dropout=0.1,
    channel_drop_prob=0.1
)

metrics = train_and_evaluate_classification(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    y_train=y_train,
    num_epochs=40,
    lr=1e-4,
    weight_decay=1e-4,
    checkpoint_path="ArchitectureV1_best.pth",  # new name to avoid clash
    device=None
)

print("Final test metrics:", metrics)


Using device: cuda
Detected 2 classes: [0 1]


Epoch 001 | TrainLoss=0.6929 | ValLoss=0.6935 | ValACC=47.63% | ValF1(binary)=0.645
  👉 New best model saved (ValF1=0.645)


Epoch 002 | TrainLoss=0.6861 | ValLoss=0.6926 | ValACC=47.63% | ValF1(binary)=0.645


Epoch 003 | TrainLoss=0.6615 | ValLoss=0.6650 | ValACC=50.00% | ValF1(binary)=0.656
  👉 New best model saved (ValF1=0.656)


Epoch 004 | TrainLoss=0.5901 | ValLoss=0.5392 | ValACC=79.62% | ValF1(binary)=0.819
  👉 New best model saved (ValF1=0.819)


...

Epoch 037 | TrainLoss=0.1874 | ValLoss=0.7085 | ValACC=78.67% | ValF1(binary)=0.804


Epoch 038 | TrainLoss=0.1874 | ValLoss=0.7194 | ValACC=78.91% | ValF1(binary)=0.807


Epoch 039 | TrainLoss=0.1804 | ValLoss=0.4426 | ValACC=81.28% | ValF1(binary)=0.792


Epoch 040 | TrainLoss=0.1832 | ValLoss=2.9890 | ValACC=48.34% | ValF1(binary)=0.648

Loading best model from: ArchitectureV1_best.pth



TestACC=73.34% | TestF1(binary)=0.740
Final test metrics: {'ACC': 0.7334494773519163, 'F1': 0.7402376910016978}


In [12]:
import numpy as np

# Number of classes from labels
n_classes = len(np.unique(y_train))
model = SpatioSpectroTemporalNet(
    d_model=64,
    temporal_hidden=64,
    num_layers_rnn=1,
    num_classes=n_classes,   # make sure this matches your labels
    use_bi=True,
    dropout=0.1,
    channel_drop_prob=0.1
)

metrics = train_and_evaluate_classification(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    y_train=y_train,
    num_epochs=40,
    lr=1e-4,
    weight_decay=1e-4,
    checkpoint_path="ArchitectureV1_best(FLoss).pth",  # new name to avoid clash
    device=None
)

print("Final test metrics:", metrics)


Using device: cuda
Detected 2 classes: [0 1]


Epoch 001 | TrainLoss=0.1736 | ValLoss=0.1833 | ValACC=52.37% | ValF1(binary)=0.000


Epoch 002 | TrainLoss=0.1620 | ValLoss=0.1593 | ValACC=66.59% | ValF1(binary)=0.528
  👉 New best model saved (ValF1=0.528)


Epoch 003 | TrainLoss=0.1451 | ValLoss=0.1300 | ValACC=78.20% | ValF1(binary)=0.758
  👉 New best model saved (ValF1=0.758)


Epoch 004 | TrainLoss=0.1343 | ValLoss=0.1273 | ValACC=80.81% | ValF1(binary)=0.816
  👉 New best model saved (ValF1=0.816)


...

Epoch 037 | TrainLoss=0.0441 | ValLoss=0.2598 | ValACC=80.33% | ValF1(binary)=0.817


Epoch 038 | TrainLoss=0.0436 | ValLoss=0.7894 | ValACC=58.53% | ValF1(binary)=0.697


Epoch 039 | TrainLoss=0.0444 | ValLoss=0.5876 | ValACC=64.45% | ValF1(binary)=0.726


Epoch 040 | TrainLoss=0.0473 | ValLoss=0.1567 | ValACC=79.15% | ValF1(binary)=0.772

Loading best model from: ArchitectureV1_best(FLoss).pth



TestACC=75.61% | TestF1(binary)=0.757
Final test metrics: {'ACC': 0.7560975609756098, 'F1': 0.7569444444444444}


In [31]:
import random
import numpy as np
import torch

# 1) Helper: create a fresh model each time
def create_model():
    model = SpatioSpectroTemporalNet(
        d_model=64,
        temporal_hidden=32,
        num_layers_rnn=1,
        num_classes=2,
        use_bi=True,
        dropout=0.1,
        channel_drop_prob=0.1,   # or whatever you used
    )
    return model

# 2) Run multiple seeds
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
results = []

for seed in seeds:
    print(f"\n===== SEED {seed} =====")

    # Fix all RNGs
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # New model
    model = create_model()

    # Seed-specific checkpoint name
    ckpt_path = f"ArchitectureV1_FLoss_seed{seed}.pth"

    metrics = train_and_evaluate_classification(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        y_train=y_train,
        num_epochs=40,        # or your chosen epochs
        lr=1e-4,
        weight_decay=1e-4,
        checkpoint_path=ckpt_path,
        device=None
    )

    results.append({
        "seed": seed,
        "ACC": metrics["ACC"],
        "F1":  metrics["F1"],
        "ckpt": ckpt_path,
    })

# 3) Compute mean ± std across seeds
accs = np.array([r["ACC"] for r in results])
f1s  = np.array([r["F1"]  for r in results])

mean_acc = accs.mean()
std_acc  = accs.std()
mean_f1  = f1s.mean()
std_f1   = f1s.std()

print("\n===== Summary over seeds =====")
for r in results:
    print(f"Seed {r['seed']}: ACC={r['ACC']*100:.2f}%  F1={r['F1']:.3f}  ckpt={r['ckpt']}")

print(f"\nMean ACC = {mean_acc*100:.2f}% ± {std_acc*100:.2f}%")
print(f"Mean F1  = {mean_f1:.3f} ± {std_f1:.3f}")

# 4) Pick representative seed (closest to mean in ACC+F1 space)
distances = (accs - mean_acc)**2 + (f1s - mean_f1)**2
best_idx = distances.argmin()
rep = results[best_idx]

print("\n===== Representative model for GradCAM =====")
print(f"Seed {rep['seed']}  |  ACC={rep['ACC']*100:.2f}%  F1={rep['F1']:.3f}")
print(f"Use checkpoint: {rep['ckpt']}  for Grad-CAM and analysis.")



===== SEED 0 =====
Using device: cuda
Detected 2 classes: [0 1]


Epoch 001 | TrainLoss=0.1771 | ValLoss=0.1798 | ValACC=50.77% | ValF1(binary)=0.000


Epoch 002 | TrainLoss=0.1662 | ValLoss=0.1655 | ValACC=70.69% | ValF1(binary)=0.637
  👉 New best model saved (ValF1=0.637)


Epoch 003 | TrainLoss=0.1530 | ValLoss=0.1552 | ValACC=71.07% | ValF1(binary)=0.689
  👉 New best model saved (ValF1=0.689)


Epoch 004 | TrainLoss=0.1452 | ValLoss=0.1483 | ValACC=72.61% | ValF1(binary)=0.710
  👉 New best model saved (ValF1=0.710)


...

Epoch 037 | TrainLoss=0.0790 | ValLoss=0.1324 | ValACC=80.84% | ValF1(binary)=0.791


Epoch 038 | TrainLoss=0.0772 | ValLoss=0.1201 | ValACC=76.05% | ValF1(binary)=0.694


Epoch 039 | TrainLoss=0.0779 | ValLoss=0.3568 | ValACC=75.48% | ValF1(binary)=0.792


Epoch 040 | TrainLoss=0.0744 | ValLoss=0.5787 | ValACC=64.18% | ValF1(binary)=0.731

Loading best model from: ArchitectureV1_FLoss_seed9.pth



TestACC=76.72% | TestF1(binary)=0.739

===== Summary over seeds =====
Seed 0: ACC=76.15%  F1=0.730  ckpt=ArchitectureV1_FLoss_seed0.pth
Seed 1: ACC=76.87%  F1=0.770  ckpt=ArchitectureV1_FLoss_seed1.pth
Seed 2: ACC=73.71%  F1=0.747  ckpt=ArchitectureV1_FLoss_seed2.pth
Seed 3: ACC=75.86%  F1=0.711  ckpt=ArchitectureV1_FLoss_seed3.pth
Seed 4: ACC=71.98%  F1=0.737  ckpt=ArchitectureV1_FLoss_seed4.pth
Seed 5: ACC=77.73%  F1=0.744  ckpt=ArchitectureV1_FLoss_seed5.pth
Seed 6: ACC=77.01%  F1=0.735  ckpt=ArchitectureV1_FLoss_seed6.pth
Seed 7: ACC=75.14%  F1=0.748  ckpt=ArchitectureV1_FLoss_seed7.pth
Seed 8: ACC=76.15%  F1=0.774  ckpt=ArchitectureV1_FLoss_seed8.pth
Seed 9: ACC=76.72%  F1=0.739  ckpt=ArchitectureV1_FLoss_seed9.pth

Mean ACC = 75.73% ± 1.64%
Mean F1  = 0.743 ± 0.017

===== Representative model for GradCAM =====
Seed 7  |  ACC=75.14%  F1=0.748
Use checkpoint: ArchitectureV1_FLoss_seed7.pth  for Grad-CAM and analysis.


In [29]:
import random
import numpy as np
import torch

# 1) Helper: create a fresh model each time
def create_model():
    model = SpatioSpectroTemporalNet(
        d_model=64,
        temporal_hidden=32,
        num_layers_rnn=1,
        num_classes=2,
        use_bi=True,
        dropout=0.1,
        channel_drop_prob=0.1,   # or whatever you used
    )
    return model

# 2) Run multiple seeds
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
results = []

for seed in seeds:
    print(f"\n===== SEED {seed} =====")

    # Fix all RNGs
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # New model
    model = create_model()

    # Seed-specific checkpoint name
    ckpt_path = f"ArchitectureV1_CELoss_seed{seed}.pth"

    metrics = train_and_evaluate_classification(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        y_train=y_train,
        num_epochs=40,        # or your chosen epochs
        lr=1e-4,
        weight_decay=1e-4,
        checkpoint_path=ckpt_path,
        device=None
    )

    results.append({
        "seed": seed,
        "ACC": metrics["ACC"],
        "F1":  metrics["F1"],
        "ckpt": ckpt_path,
    })

# 3) Compute mean ± std across seeds
accs = np.array([r["ACC"] for r in results])
f1s  = np.array([r["F1"]  for r in results])

mean_acc = accs.mean()
std_acc  = accs.std()
mean_f1  = f1s.mean()
std_f1   = f1s.std()

print("\n===== Summary over seeds =====")
for r in results:
    print(f"Seed {r['seed']}: ACC={r['ACC']*100:.2f}%  F1={r['F1']:.3f}  ckpt={r['ckpt']}")

print(f"\nMean ACC = {mean_acc*100:.2f}% ± {std_acc*100:.2f}%")
print(f"Mean F1  = {mean_f1:.3f} ± {std_f1:.3f}")

# 4) Pick representative seed (closest to mean in ACC+F1 space)
distances = (accs - mean_acc)**2 + (f1s - mean_f1)**2
best_idx = distances.argmin()
rep = results[best_idx]

print("\n===== Representative model for GradCAM =====")
print(f"Seed {rep['seed']}  |  ACC={rep['ACC']*100:.2f}%  F1={rep['F1']:.3f}")
print(f"Use checkpoint: {rep['ckpt']}  for Grad-CAM and analysis.")



===== SEED 0 =====
Using device: cuda
Detected 2 classes: [0 1]


Epoch 001 | TrainLoss=0.1771 | ValLoss=0.1798 | ValACC=50.77% | ValF1(binary)=0.000


Epoch 002 | TrainLoss=0.1662 | ValLoss=0.1655 | ValACC=70.69% | ValF1(binary)=0.637
  👉 New best model saved (ValF1=0.637)


Epoch 003 | TrainLoss=0.1530 | ValLoss=0.1552 | ValACC=71.07% | ValF1(binary)=0.689
  👉 New best model saved (ValF1=0.689)


Epoch 004 | TrainLoss=0.1452 | ValLoss=0.1483 | ValACC=72.61% | ValF1(binary)=0.710
  👉 New best model saved (ValF1=0.710)


...

Epoch 037 | TrainLoss=0.0790 | ValLoss=0.1324 | ValACC=80.84% | ValF1(binary)=0.791


Epoch 038 | TrainLoss=0.0772 | ValLoss=0.1201 | ValACC=76.05% | ValF1(binary)=0.694


Epoch 039 | TrainLoss=0.0779 | ValLoss=0.3568 | ValACC=75.48% | ValF1(binary)=0.792


Epoch 040 | TrainLoss=0.0744 | ValLoss=0.5787 | ValACC=64.18% | ValF1(binary)=0.731

Loading best model from: ArchitectureV1_CELoss_seed9.pth



TestACC=76.72% | TestF1(binary)=0.739

===== Summary over seeds =====
Seed 0: ACC=76.15%  F1=0.730  ckpt=ArchitectureV1_CELoss_seed0.pth
Seed 1: ACC=76.87%  F1=0.770  ckpt=ArchitectureV1_CELoss_seed1.pth
Seed 2: ACC=73.71%  F1=0.747  ckpt=ArchitectureV1_CELoss_seed2.pth
Seed 3: ACC=75.86%  F1=0.711  ckpt=ArchitectureV1_CELoss_seed3.pth
Seed 4: ACC=71.98%  F1=0.737  ckpt=ArchitectureV1_CELoss_seed4.pth
Seed 5: ACC=77.73%  F1=0.744  ckpt=ArchitectureV1_CELoss_seed5.pth
Seed 6: ACC=77.01%  F1=0.735  ckpt=ArchitectureV1_CELoss_seed6.pth
Seed 7: ACC=75.14%  F1=0.748  ckpt=ArchitectureV1_CELoss_seed7.pth
Seed 8: ACC=76.15%  F1=0.774  ckpt=ArchitectureV1_CELoss_seed8.pth
Seed 9: ACC=76.72%  F1=0.739  ckpt=ArchitectureV1_CELoss_seed9.pth

Mean ACC = 75.73% ± 1.64%
Mean F1  = 0.743 ± 0.017

===== Representative model for GradCAM =====
Seed 7  |  ACC=75.14%  F1=0.748
Use checkpoint: ArchitectureV1_CELoss_seed7.pth  for Grad-CAM and analysis.
